# 03 — Quantum Kernels

Visualize the three feature-map circuits we use, then compute their
kernel matrices on a small slice of the training data so we can
inspect the structure that QSVM will fit on.

Feature maps:
* **ZZFeatureMap** — depth-1/2 entangling Hadamard + ZZ rotations.
* **PauliFeatureMap** — Z, ZZ, ZZZ Pauli rotations.
* **Custom** — Hadamard + RY/RZ ring-entangled, with pairwise RZ couplings.

In [ ]:
%matplotlib inline
import warnings

import matplotlib.pyplot as plt
import numpy as np

from qml_healthcare.config import DEFAULT_QUBITS, DEFAULT_REPS
from qml_healthcare.data.preprocess import prepare_data
from qml_healthcare.evaluation import figure_path, plot_kernel_heatmap
from qml_healthcare.models.quantum_kernels import (
    FEATURE_MAP_NAMES,
    build_feature_map,
    compute_kernel_matrix,
    make_quantum_kernel,
)

warnings.filterwarnings("ignore")

In [ ]:
n_features = DEFAULT_QUBITS
reps = DEFAULT_REPS
for name in FEATURE_MAP_NAMES:
    qc = build_feature_map(name, n_features=n_features, reps=reps)
    print(f"\n=== {name} (qubits={qc.num_qubits}, depth={qc.decompose().depth()}) ===")
    fig = qc.draw("mpl", fold=-1)
    fig.savefig(figure_path(f"feature_map_{name}.png"), dpi=140, bbox_inches="tight")
    plt.show()

In [ ]:
bundle = prepare_data()
n_viz = 60
X_viz = bundle.X_train_q[:n_viz]
y_viz = bundle.y_train_q[:n_viz]
print(f"Quantum subset: N={len(bundle.X_train_q)}, K={bundle.n_quantum_features}")
print(f"Quantum features: {bundle.quantum_feature_names}")

In [ ]:
for name in FEATURE_MAP_NAMES:
    qc = build_feature_map(name, n_features=bundle.n_quantum_features, reps=reps)
    kernel = make_quantum_kernel(qc)
    K = compute_kernel_matrix(kernel, X_viz)
    plot_kernel_heatmap(
        K, y_viz, figure_path(f"kernel_heatmap_{name}.png"), f"Quantum kernel ({name}) — N={n_viz}"
    )
    eigs = np.linalg.eigvalsh(K)
    print(
        f"  {name}: K shape={K.shape}, off-diag mean={(K - np.diag(np.diag(K))).mean():.4f}, "
        f"eig range=[{eigs.min():.3f}, {eigs.max():.3f}]"
    )

**Reading the heatmaps**: a useful kernel will look block-diagonal
after rows/cols are reordered by class label — same-class fidelities
should be high, cross-class fidelities low. If a heatmap looks like
noise, that feature map is not separating the classes.